# 05 · Multi-agent patterns

Implement the multi-agent patterns interviewers name (router, supervisor/worker, sequential pipeline, shared blackboard) in plain Python with the same client, measure what each costs in model calls and tokens on the same tickets, and reproduce the failure modes (ping-pong handoffs, context lost at a handoff, cost blow-up, compounding errors) together with their guards.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · The agent loop](01_agent_loop_from_scratch.ipynb), [03 · ReAct and plan-and-execute](03_react_and_plan_and_execute.ipynb)

## Why this matters in interviews

- "How would you set up a multi-agent system?" is really "when would you *not*?" Interviewers reward scepticism with a threshold (`ag13`).
- The router is the highest-leverage pattern for cost (`ag31`); the supervisor is the most common topology (`ag27`).
- "Five specialists score worse than one generalist" (`ag46`) has a precise answer: lossy handoffs, compounding errors and unrecoverable routing.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag11` | What are the main agentic design patterns? |
| `ag12` | How do you set up a multi-agent system, and when should you? |
| `ag13` | What goes wrong in multi-agent systems? |
| `ag27` | What is a supervisor agent and how does it decide what to delegate? |
| `ag31` | What is a router agent, and why is it the highest-leverage pattern? |
| `ag32` | How do you parallelise agent work safely? |
| `ag46` | You built five specialised agents. Together they score worse than one general agent on the same base model. Why? |

## 1. The topologies

```
Router               Supervisor / worker        Sequential pipeline            Blackboard
                                                                                 +-----------------+
ticket -> [router]   ticket -> [supervisor] <-+  ticket -> [research] ->        | shared state    |
            |   \                |      \     |             [write] ->          | ticket, category|
       [billing] [tech]      [worker A] [worker B]          [edit] -> reply     | facts, reply    |
       (one specialist)      results go back to the                             +-----------------+
                             supervisor, which answers                           agents read/write it;
                                                                                 a controller picks who runs
```

| Pattern | Who decides what happens next | Communication | Use it when | Main risk |
|---|---|---|---|---|
| **router** | a cheap classifier, once | none after routing | requests fall into known kinds | a confidently wrong route nobody can undo |
| **supervisor / worker** | the supervisor, every step | briefs down, results up | subtasks need different tools or prompts | the supervisor's context becomes the bottleneck |
| **sequential pipeline** | you, at design time | fixed handoffs | the stages are always the same | lossy handoffs, errors compound |
| **peer handoff** | each agent | agents transfer control | conversational specialists (e.g. triage → billing) | ping-pong loops, hard to bound |
| **debate / critique** | a judge | proposals and critiques | a checkable decision with real trade-offs | 3x+ the calls; judges have biases |
| **blackboard** | a controller reading shared state | one structured state | many contributors, inspectable progress | needs a clear schema and a stop rule |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

import re
import json
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

random.seed(0)
rng = np.random.default_rng(0)


class Meter:
    """Every model call goes through here, so we can count calls and tokens per pattern."""
    def __init__(self):
        self.calls, self.tokens = defaultdict(int), defaultdict(int)

    def chat(self, pattern, messages, **kw):
        resp = client.chat.completions.create(model=MODEL, messages=messages, **kw)
        self.calls[pattern] += 1
        self.tokens[pattern] += resp.usage.total_tokens
        return resp.choices[0].message


meter = Meter()

## 2. The shared setup: three tickets, three specialists

Each specialist is just a system prompt with its own policy knowledge (in a real system: its own tools and permissions). Offline, the stand-in answers a specialist by quoting the policy sentence that best matches the ticket; a real model writes a proper reply from the same policy.

In [ ]:
TICKETS = ["I was billed twice this month for order ORD-448812. Please refund the duplicate charge.",
           "The app shows an error when I log in on my phone.",
           "My package has not been delivered yet; it shipped a week ago."]
EXPECTED = ["billing", "bug", "delivery"]

SPECIALISTS = {
    "billing": "You are the billing specialist. Policy: Duplicate charges are refunded in full within 5 business days. "
               "Refunds go back to the original payment method. Invoices can be downloaded from the account page. "
               "Answer the customer in one or two sentences.",
    "bug": "You are the technical support specialist. Known issue: Login errors on the mobile app are fixed by "
           "updating to version 4.2. Clearing the app cache fixes most other errors. Crashes should be reported "
           "with the device model. Answer the customer in one or two sentences.",
    "delivery": "You are the delivery specialist. Policy: Packages not delivered within 7 days of shipping are traced "
                "with the carrier. A replacement is sent if the trace fails. Tracking updates can lag by 24 hours. "
                "Answer the customer in one or two sentences.",
}


def specialist(name, text, pattern):
    return meter.chat(pattern, [{"role": "system", "content": SPECIALISTS[name]}, {"role": "user", "content": text}]).content

### Baseline: one agent with all the knowledge as tools

Before any multi-agent design, measure the simplest thing: one agent (the loop from notebook 01) with one search tool per knowledge area.

In [ ]:
def kb_tool(name, description):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}


KB_TOOLS = [kb_tool("billing_policy", "Billing policy: charges, billed twice, refunds, invoices, payments."),
            kb_tool("tech_policy", "Technical fixes: app errors, login problems, crashes, bugs."),
            kb_tool("delivery_policy", "Delivery policy: packages, shipping, tracking, late or missing parcels.")]
def policy_only(prompt):
    """The policy sentences of a specialist prompt, without the role and instructions."""
    return prompt.split(". ", 1)[1].rsplit(" Answer the customer", 1)[0]


POLICY_TEXT = {"billing_policy": policy_only(SPECIALISTS["billing"]), "tech_policy": policy_only(SPECIALISTS["bug"]),
               "delivery_policy": policy_only(SPECIALISTS["delivery"])}


def single_agent(ticket, max_steps=4):
    msgs = [{"role": "system", "content": "You are a support agent. Look up the policy, then answer briefly."},
            {"role": "user", "content": ticket}]
    for _ in range(max_steps):
        m = meter.chat("single agent", msgs, tools=KB_TOOLS)
        if not m.tool_calls:
            return m.content
        msgs.append({"role": "assistant", "content": m.content, "tool_calls": [t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": POLICY_TEXT[t.function.name]})
    return None


for t in TICKETS:
    print(f"{t[:45]:45s} -> {single_agent(t)[:90]}")

## 3. Router: classify once, then one specialist

The router is a cheap, single decision at the front. It is the highest-leverage pattern because most traffic is easy: route it to a cheap specialist (or model tier) and reserve the expensive path for what needs it (`ag31`). Notebook [08](08_system1_decision_models_jev_and_laya.ipynb) replaces this LLM router with a calibrated classifier that answers in milliseconds.

In [ ]:
ROUTER_SYSTEM = "Route the customer ticket. Reply with one of: billing, bug, delivery."


def route(ticket, pattern="router"):
    label = meter.chat(pattern, [{"role": "system", "content": ROUTER_SYSTEM},
                                  {"role": "user", "content": f'Ticket: "{ticket}"'}]).content.strip().lower()
    return label if label in SPECIALISTS else "fallback"


def router_pattern(ticket):
    label = route(ticket)
    if label == "fallback":
        return label, "Let me pass you to a human colleague."        # never guess on an unknown route
    return label, specialist(label, ticket, "router")


routes = []
for t in TICKETS:
    label, reply = router_pattern(t)
    routes.append(label)
    print(f"[{label:8s}] {reply}")
print("router accuracy:", np.mean([r == e for r, e in zip(routes, EXPECTED)]))
assert routes == EXPECTED

All three routed correctly here. Routers **do** misroute, and a misrouted ticket never comes back (the billing specialist cannot fix a login bug). So measure router accuracy on labelled traffic, send unknown labels to a fallback rather than guessing, and track the escalation rate per route.

## 4. Supervisor / worker: delegate as tool calls

The supervisor owns the goal. Workers are exposed to it **as tools**; each worker call runs a separate model call with its own system prompt and only the brief it was given. The supervisor sees workers' *outputs*, not their contexts, which keeps its own window small (`ag27`).

In [ ]:
def worker_tool(name, description):
    return {"type": "function", "function": {"name": f"{name}_worker", "description": description, "parameters": {
        "type": "object", "properties": {"task": {"type": "string", "description": "the part of the ticket to handle"}},
        "required": ["task"]}}}


WORKER_TOOLS = [worker_tool("billing", "Billing specialist: invoices, charges, billed twice, refunds, payments."),
                worker_tool("bug", "Technical specialist: app errors, login problems, crashes, bugs."),
                worker_tool("delivery", "Delivery specialist: packages, shipping, tracking, late or missing parcels.")]
SUPERVISOR_SYSTEM = "You are the support supervisor. Delegate each part of the ticket to the right worker, then answer."


def supervisor(ticket, max_steps=4, pattern="supervisor"):
    msgs = [{"role": "system", "content": SUPERVISOR_SYSTEM}, {"role": "user", "content": ticket}]
    delegated = []
    for _ in range(max_steps):
        m = meter.chat(pattern, msgs, tools=WORKER_TOOLS)
        if not m.tool_calls:
            return m.content, delegated
        msgs.append({"role": "assistant", "content": m.content, "tool_calls": [t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            name = t.function.name.removesuffix("_worker")
            delegated.append(name)
            result = specialist(name, json.loads(t.function.arguments)["task"], pattern)
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": result})
    return None, delegated


for t in TICKETS:
    answer, delegated = supervisor(t)
    print(f"delegated to {delegated} -> {answer[:80]}")

answer, delegated = supervisor("I was billed twice for order ORD-448812 and want the duplicate charge refunded, "
                               "and my package has not been delivered yet.", pattern="supervisor (2-issue ticket)")
print("\n2-issue ticket: delegated to", delegated)
print("answer:", answer)
assert sorted(delegated) == ["billing", "delivery"]

On the two-issue ticket the supervisor delegated to two workers in parallel and then combined their results; offline the combination is a quote of both results, a real model writes one reply. Design decisions worth naming (`ag27`): does the supervisor see workers' full context or only their outputs (usually only outputs), can workers ask clarifying questions, what is the delegation depth limit, and is the brief free text or a **typed schema** (section 7 shows why typed wins).

## 5. Sequential pipeline: research → write → edit

Fixed stages, each specialised. Barely an "agent" system, and more predictable for it.

In [ ]:
KB_ALL = "\n".join(f"[{k}] {v}" for k, v in SPECIALISTS.items())


def pipeline(ticket):
    notes = meter.chat("pipeline", [{"role": "system", "content": "You are a researcher. Find the policy facts relevant "
                                     f"to the ticket and cite their tags.\n\nKnowledge base:\n{KB_ALL}"},
                                    {"role": "user", "content": ticket}]).content
    draft = meter.chat("pipeline", [{"role": "system", "content": "You are a writer. Write a short, friendly reply "
                                     "using only the research notes. Always thank the customer first. Never promise "
                                     "dates or amounts that are not in the notes. Keep the reply under 60 words."},
                                    {"role": "user", "content": f"Research notes: {notes}\n\nQuestion: {ticket}"}]).content
    final = meter.chat("pipeline", [{"role": "system", "content": "You are an editor for customer replies."},
                                    {"role": "user", "content": "Summarize this draft in at most two sentences, "
                                     f"keeping every fact and number:\n\n{draft}"}]).content
    return notes, draft, final


notes, draft, final = pipeline(TICKETS[0])
print("research:", notes, "\ndraft   :", draft, "\nfinal   :", final)
for t in TICKETS[1:]:
    pipeline(t)

Each stage only sees what the previous stage wrote. That is the pipeline's strength (small, focused prompts) and its weakness: anything a stage drops is gone for every later stage.

## 6. Blackboard: shared state and a controller

Instead of passing messages, agents read and write **one structured state**. A controller repeatedly picks the first agent whose inputs are ready and whose output is still empty. Every write is logged, so progress is inspectable and the whole state can be checkpointed.

In [ ]:
ORDERS = {"ORD-448812": {"status": "DELIVERED", "charged_twice": True}}


def bb_classify(b):
    return route(b["ticket"], pattern="blackboard")


def bb_order(b):                                     # a tool, not a model call
    m = re.search(r"ORD-\d{6}", b["ticket"])
    return ORDERS.get(m.group(0), "not found") if m else "no order id"


def bb_policy(b):
    return specialist(b["category"], b["ticket"], "blackboard")


def bb_write(b):
    return meter.chat("blackboard", [{"role": "system", "content": "Write a short reply to the customer from the facts. "
                                     "Thank the customer first. Never promise anything that is not in the facts. Keep it under 60 words."},
                                     {"role": "user", "content": f"Facts: {b['policy']} Order: {b['order']}\n\nQuestion: {b['ticket']}"}]).content


BB_AGENTS = [("classifier", ["ticket"], "category", bb_classify),
             ("order_lookup", ["category"], "order", bb_order),
             ("policy", ["category"], "policy", bb_policy),
             ("writer", ["policy", "order"], "reply", bb_write)]


def blackboard(ticket, max_rounds=8):
    board = {"ticket": ticket, "category": None, "order": None, "policy": None, "reply": None}
    log = []
    for _ in range(max_rounds):
        ready = [(name, out, fn) for name, needs, out, fn in BB_AGENTS
                 if board[out] is None and all(board[n] is not None for n in needs)]
        if not ready or board["reply"] is not None:
            break
        name, out, fn = ready[0]
        board[out] = fn(board)
        log.append(f"{name:12s} wrote {out:8s} = {str(board[out])[:60]}")
    return board, log


board, log = blackboard(TICKETS[0])
print("\n".join(log))
assert board["reply"] is not None and board["category"] == "billing"
for t in TICKETS[1:]:
    blackboard(t)

## 7. What each pattern costs

Every pattern above handled the same three tickets (the supervisor also got the two-issue ticket, counted separately). Same model, same knowledge; the only difference is the architecture.

In [ ]:
patterns = ["single agent", "router", "supervisor", "pipeline", "blackboard"]
calls = [meter.calls[p] / len(TICKETS) for p in patterns]
toks = [meter.tokens[p] / len(TICKETS) for p in patterns]
for p, c, t in zip(patterns, calls, toks):
    print(f"{p:14s} {c:.1f} model calls, {t:6.0f} tokens per ticket")
print(f"{'supervisor, 2-issue ticket':14s}: {meter.calls['supervisor (2-issue ticket)']} calls")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].bar(patterns, calls, color="#4c6ef5"); axes[0].set_ylabel("model calls per ticket")
axes[1].bar(patterns, toks, color="#fab005"); axes[1].set_ylabel("tokens per ticket")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Same tickets, same model: the architecture sets the bill")
plt.tight_layout(); plt.show()
assert meter.calls["router"] <= meter.calls["pipeline"] and meter.calls["single agent"] < meter.calls["supervisor"] + 1

The router is the cheapest design that still uses a specialist; the supervisor pays for its own reasoning on top of every worker call; the pipeline pays for every stage on every ticket, whether the ticket needs it or not. With real models the gap widens: a supervisor re-reads the growing history on every step, and workers often need several steps each.

## 8. Failure modes and their guards

### 8a. Ping-pong handoffs

Two peer agents, each convinced the other should handle "I get an error when I try to pay my bill": billing sees "error" and hands off to tech; tech sees "pay" and hands back. The rules below are deterministic so the loop is easy to see.

In [ ]:
def billing_peer(ticket):
    return ("handoff", "tech") if "error" in ticket else ("answer", "billing handled it")


def tech_peer(ticket):
    return ("handoff", "billing") if "pay" in ticket else ("answer", "tech handled it")


PEERS = {"billing": billing_peer, "tech": tech_peer}


def run_handoffs(ticket, start="billing", max_handoffs=None, detect_cycles=False):
    current, trail, seen = start, [start], set()
    while True:
        kind, value = PEERS[current](ticket)
        if kind == "answer":
            return "answered", trail
        if detect_cycles and (current, value) in seen:
            return "cycle detected -> escalate to supervisor/human", trail
        seen.add((current, value))
        if max_handoffs is not None and len(trail) > max_handoffs:
            return "max handoffs -> escalate", trail
        current = value
        trail.append(current)
        if len(trail) > 50:
            return "runaway (would loop forever)", trail


ticket = "I get an error when I try to pay my bill."
for kwargs in [{}, {"max_handoffs": 4}, {"detect_cycles": True}]:
    status, trail = run_handoffs(ticket, **kwargs)
    print(f"{str(kwargs):26s} {status:48s} {' -> '.join(trail[:6])}{' ...' if len(trail) > 6 else ''}")

Guards: a **handoff depth limit**, a **cycle detector** (the same agent-to-agent handoff twice), and an owner who breaks ties (a supervisor or a human). The deeper fix is design: overlapping responsibilities cause ping-pong, so give each agent a clear "I own X" boundary and one agent ownership of the final answer.

### 8b. Context lost at the handoff

A triage agent passes a conversation to the billing agent. As prose, the handoff is a summary; as a **typed schema**, every field the next agent needs is required.

In [ ]:
CONVERSATION = ("customer: Hi, this is Ana Lima. I was charged $89.20 twice for the same kettle. "
                "agent: Sorry to hear that, let me check. customer: The order id is ORD-448812 and I paid by card.")

prose = meter.chat("handoffs", [{"role": "user", "content": f"Summarize this conversation for the billing team:\n\n{CONVERSATION}"}]).content


class BillingHandoff(BaseModel):
    customer_name: str
    order_id: str = Field(pattern=r"^ORD-\d{6}$")
    amount_usd: float
    description: str = Field(description="one sentence on what the customer needs")


typed = client.chat.completions.parse(model=MODEL, response_format=BillingHandoff, messages=[
    {"role": "system", "content": "Extract the handoff for the billing team."},
    {"role": "user", "content": CONVERSATION}]).choices[0].message.parsed

print("prose handoff :", prose)
print("typed handoff :", typed.model_dump(exclude={"description"}))
print("billing agent can find the order? prose:", "ORD-448812" in prose, "| typed:", typed.order_id == "ORD-448812")
assert "ORD-448812" not in prose and typed.order_id == "ORD-448812"

The prose summary kept the gist and dropped the order id, so the billing agent has to ask the customer again (or guess). The typed handoff cannot drop it: `order_id` is required and validated by a pattern. This is the fix for most multi-agent quality loss (`ag12`, `ag46`): **pass artefacts, not summaries**, as typed schemas or references to shared state.

### 8c. Cost blow-up, and a budget shared by every agent

Calls multiply: a supervisor that takes `S` steps and delegates to `W` workers per step, each taking `K` steps, makes `S × (1 + W × K)` model calls.

In [ ]:
S = np.arange(1, 7)
fig, ax = plt.subplots(figsize=(8, 3.4))
for W, K in [(1, 1), (2, 2), (3, 3)]:
    ax.plot(S, S * (1 + W * K), "o-", label=f"{W} workers/step, {K} steps each")
ax.axhline(2, color="gray", ls="--", label="single agent, one tool round")
ax.set_xlabel("supervisor steps"); ax.set_ylabel("model calls per request")
ax.set_title("Multi-agent cost multiplies: calls = S x (1 + W x K)")
ax.legend(fontsize=8)
plt.show()


class SharedBudget:
    def __init__(self, max_calls):
        self.max_calls, self.used = max_calls, 0

    def charge(self):
        if self.used >= self.max_calls:
            raise RuntimeError(f"shared budget of {self.max_calls} model calls exhausted")
        self.used += 1


budget = SharedBudget(max_calls=3)
_chat = meter.chat


def budgeted_chat(pattern, messages, **kw):
    budget.charge()                                   # checked BEFORE the call, by every agent
    return _chat(pattern, messages, **kw)


meter.chat = budgeted_chat
try:
    supervisor("I was billed twice for order ORD-448812, and my package has not been delivered yet.", pattern="budgeted")
except RuntimeError as e:
    print("stopped:", e, f"(after {budget.used} calls)")
finally:
    meter.chat = _chat
assert budget.used == 3

One budget object, charged by the supervisor *and* every worker before each call, bounds the whole tree. Per-agent budgets do not: five agents each within budget can still blow the request budget together.

### 8d. Errors compound

If each hop in a chain is right 90% of the time, four hops are right about 66% of the time.

In [ ]:
hops = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(8, 3.4))
for p in (0.99, 0.95, 0.90, 0.80):
    ax.plot(hops, p ** hops * 100, "o-", label=f"{p:.0%} per agent")
ax.set_xlabel("agents in the chain"); ax.set_ylabel("end-to-end success (%)")
ax.set_title("Independent per-hop errors multiply: success = p^n")
ax.legend(fontsize=8)
plt.show()
print(f"0.9^4 = {0.9 ** 4:.2f}, 0.9^5 = {0.9 ** 5:.2f}")

This is the core of `ag46`: five specialists can each beat the generalist on their own slice and the chain still loses, because the losses multiply and nothing downstream can recover information an upstream hop dropped. A generalist that starts down the wrong path can notice and turn round; a misrouted task in a fan-out never comes back.

## 9. When is multi-agent worth it?

| Situation | Multi-agent? | Why |
|---|---|---|
| subtasks need different tools, permissions or data (a code agent vs a customer-email agent) | yes | isolation and least privilege; contexts do not collide |
| independent, read-only breadth work (search many sources, review many files) | yes, fan-out | parallel speed-up; nothing shared to corrupt |
| each specialist needs its own evaluation and owner | yes | you can test and improve them separately |
| the work shares mutable state (one codebase, one order, one document) | usually no | agents form contradictory beliefs; conflicts |
| one agent with 5-10 well-described tools handles it | no | fewer handoffs, cheaper, one trajectory to evaluate |
| you cannot name the failure that splitting would fix | no | the default answer (`ag13`) |

**Parallelising safely (`ag32`):** only independent, read-only steps; express dependencies as a graph; cap concurrency; decide in advance what partial failure means, and *tell the model which branches are missing* rather than letting it assume the results it got were all there were.

## Try it yourself

**Exercise 1.** Fan out three read-only lookups in parallel with a thread pool, where one branch times out. Return the results **and** an explicit list of missing branches, and build the note the model should see.

In [ ]:
# Solution — try it yourself first, then run this
import time
from concurrent.futures import ThreadPoolExecutor, wait


def lookup(source):
    time.sleep(0.5 if source == "carrier_api" else 0.05)       # the carrier API is slow today
    return f"{source}: ok"


sources = ["order_db", "payments", "carrier_api"]
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(lookup, s): s for s in sources}
    done, pending = wait(futures, timeout=0.2)
results = {futures[f]: f.result() for f in done}
missing = sorted(futures[f] for f in pending)
note = f"Results: {results}. MISSING (timed out, do not assume): {missing}"
print(note)
assert missing == ["carrier_api"]

**Exercise 2.** Make the router refuse to guess: add a confidence rule that sends a ticket to the `fallback` route when it mentions keywords of two different specialists (a crude ambiguity check), and test it on "My refund failed with an error".

In [ ]:
# Solution — try it yourself first, then run this
KEYWORDS = {"billing": {"bill", "billed", "refund", "charge", "charged", "invoice"},
            "bug": {"error", "crash", "login", "bug"},
            "delivery": {"package", "delivered", "shipped", "tracking", "parcel"}}


def cautious_route(ticket):
    words = set(re.findall(r"[a-z]+", ticket.lower()))
    hit = [k for k, kw in KEYWORDS.items() if words & kw]
    return "fallback" if len(hit) > 1 else route(ticket)


for t in ["My refund failed with an error", TICKETS[0]]:
    print(f"{t[:45]:45s} -> {cautious_route(t)}")
assert cautious_route("My refund failed with an error") == "fallback"

**Exercise 3.** Cost per *resolved* ticket: assume a model call costs $0.002 and these resolution rates: single agent 0.85, router 0.82, supervisor 0.88, pipeline 0.80. Using the measured calls per ticket, which design is cheapest per resolved ticket?

In [ ]:
# Solution — try it yourself first, then run this
resolution = {"single agent": 0.85, "router": 0.82, "supervisor": 0.88, "pipeline": 0.80}
cost_per_resolved = {p: meter.calls[p] / len(TICKETS) * 0.002 / r for p, r in resolution.items()}
for p, c in sorted(cost_per_resolved.items(), key=lambda kv: kv[1]):
    print(f"{p:14s} ${c:.4f} per resolved ticket")

## Say it in an interview

**30-second answer (`ag12`, `ag13`).** "I start with one agent and a router, and split only when I can name the failure that separation fixes: different tools or permissions, contexts that collide, or independent read-only work that parallelises. The usual topology is a supervisor that delegates to workers as tool calls and sees only their outputs. Handoffs are typed schemas, not prose. One budget covers the whole tree, with a delegation depth limit and a cycle detector. And I instrument per hop, because with 90% per hop, four hops is 66% end to end."

**Why five specialists lose to one generalist (`ag46`).** Lossy handoffs (summaries drop the detail that mattered), error compounding, unrecoverable routing, no shared state (contradictory beliefs), prompt dilution, and no single owner of the final answer. Fixes: pass artefacts not summaries, give one agent ownership of the answer with the original request in view, measure each hop, and collapse back to one agent unless splitting fixes a named failure. Parallel sub-agents win on independent read-only breadth; they lose on shared mutable state.

**Numbers.** Calls per request: a router is about 2, a supervisor is S × (1 + W × K). Success of a chain is p^n.

**Traps.** Multi-agent because it sounds sophisticated; free-text handoffs; per-agent budgets instead of one shared budget; no cycle detection in peer handoffs; a router whose accuracy nobody measures.

## Next

- [06 · Guardrails, HITL and failure modes](06_guardrails_hitl_and_failure_modes.ipynb) · [08 · System-1 decision models](08_system1_decision_models_jev_and_laya.ipynb) (a millisecond router).
- Frameworks: [LangGraph multi-agent supervisor](../14_langgraph/06_multi_agent_supervisor.ipynb) · [LangGraph conditional edges and loops](../14_langgraph/02_conditional_edges_and_loops.ipynb).
- [Agent evaluation](../12_evaluation/04_agent_evaluation.ipynb) (evaluate per hop and end to end) · [LiteLLM gateway and model routing](../16_production/06_litellm_gateway_and_model_routing.ipynb).
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) (chapter *Multi-agent*) and the [interview bank](../../ai_interview_prep/index.html).